# 🤖 Churn Prediction - Model Training (XGBoost)

**Goal:** Train an XGBoost classifier to predict customer churn.

**Process:**
1. Load preprocessed features from Feature Engineering step
2. Split data into train/test sets with stratification
3. Handle class imbalance with SMOTE
4. Train XGBoost with hyperparameter tuning via GridSearchCV
5. Evaluate model performance
6. Save final model as `churn_model.pkl`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('Libraries loaded successfully')

## 1. Load Preprocessed Data

In [ ]:
# Load features and target
X = pd.read_pickle('data/X_features.pkl')
y = pd.read_pickle('data/y_target.pkl')

# Load feature columns list from preprocess
preprocess = joblib.load('../backend/trained_models/preprocess.pkl')
feature_cols = preprocess['feature_cols']

print(f'Features: {X.shape}')
print(f'Target: {y.shape}')
print(f'\nTarget distribution:')
print(f'  Active (0): {(y==0).sum()}')
print(f'  Churned (1): {(y==1).sum()}')
print(f'  Churn rate: {y.mean()*100:.2f}%')

## 2. Train/Test Split

In [ ]:
# Split with stratification to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'  Active: {(y_train==0).sum()}, Churned: {(y_train==1).sum()}')
print(f'  Churn rate: {y_train.mean()*100:.2f}%')
print(f'\nTest set: {X_test.shape}')
print(f'  Active: {(y_test==0).sum()}, Churned: {(y_test==1).sum()}')
print(f'  Churn rate: {y_test.mean()*100:.2f}%')

## 3. Baseline XGBoost Model

In [ ]:
# Baseline XGBoost without tuning
baseline = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

baseline.fit(X_train, y_train)

# Predictions
y_pred_baseline = baseline.predict(X_test)
y_prob_baseline = baseline.predict_proba(X_test)[:, 1]

# Metrics
print('=== Baseline XGBoost Performance ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_baseline):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_baseline):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_baseline):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_baseline):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_baseline):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_baseline, target_names=['Active', 'Churned']))

## 4. Handling Class Imbalance with SMOTE

In [ ]:
# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {X_train.shape}')
print(f'  Active: {(y_train==0).sum()}, Churned: {(y_train==1).sum()}')
print(f'After SMOTE: {X_train_resampled.shape}')
print(f'  Active: {(y_train_resampled==0).sum()}, Churned: {(y_train_resampled==1).sum()}')

In [ ]:
# Train with SMOTE-balanced data
xgb_smote = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_smote.fit(X_train_resampled, y_train_resampled)

y_pred_smote = xgb_smote.predict(X_test)
y_prob_smote = xgb_smote.predict_proba(X_test)[:, 1]

print('=== XGBoost with SMOTE Performance ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_smote):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_smote):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_smote):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_smote):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_smote):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_smote, target_names=['Active', 'Churned']))

## 5. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}

# Create pipeline with SMOTE
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', xgb.XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False))
])

# Stratified K-Fold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Grid Search with limited combinations for speed
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print('Starting GridSearchCV (may take a few minutes)...')
grid_search.fit(X_train, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV F1-score: {grid_search.best_score_:.4f}')

In [ ]:
# Use best model
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print('=== Optimized XGBoost Performance ===')
print(f'Accuracy:  {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_best):.4f}')
print(f'Recall:    {recall_score(y_test, y_pred_best):.4f}')
print(f'F1-Score:  {f1_score(y_test, y_pred_best):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Active', 'Churned']))

## 6. Model Evaluation & Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active', 'Churned'], yticklabels=['Active', 'Churned'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
axes[1].plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc_score(y_test, y_prob_best):.3f})', color='darkorange', lw=2)
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].fill_between(fpr, tpr, alpha=0.1, color='darkorange')

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob_best)
no_skill = y_test.sum() / len(y_test)
axes[2].plot(recall, precision, label=f'XGBoost (F1 = {f1_score(y_test, y_pred_best):.3f})', color='green', lw=2)
axes[2].axhline(y=no_skill, color='k', linestyle='--', label=f'No Skill ({no_skill:.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()
axes[2].fill_between(recall, precision, alpha=0.1, color='green')

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

In [ ]:
# Extract the XGBoost model from the pipeline
xgb_model = best_model.named_steps['xgb']

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 15 Most Important Features:')
print(importance.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
sns.barplot(data=importance.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 8. Save Final Model

In [ ]:
# Save the best model (full pipeline with SMOTE + XGBoost)
joblib.dump(best_model, '../backend/trained_models/churn_model.pkl')
print('✅ churn_model.pkl saved to backend/trained_models/')

# Verify the saved model loads correctly
loaded_model = joblib.load('../backend/trained_models/churn_model.pkl')
test_pred = loaded_model.predict(X_test[:5])
print(f'\n✅ Model verification: {len(test_pred)} predictions successful')
print(f'   Sample predictions: {test_pred}')

## 9. Final Summary

In [ ]:
print("=" * 60)
print("📊 MODEL TRAINING SUMMARY")
print("=" * 60)

print(f"\nAlgorithm: XGBoost Classifier")
print(f"Training samples: {X_train.shape[0]} (after SMOTE: {X_train_resampled.shape[0]})")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"\nBest Hyperparameters:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")

print(f"\nTest Set Performance:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_best):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_best):.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_best):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_prob_best):.4f}")

print(f"\nSaved Files:")
print(f"  - ../backend/trained_models/churn_model.pkl (SMOTE + XGBoost pipeline)")
print(f"  - ../backend/trained_models/preprocess.pkl (encoders + scaler)")

print("\n" + "=" * 60)
print("✅ Model Training Complete!")
print("=" * 60)